# Deep Research Agent — Learning Implementation

This notebook documents my learning implementation from Lab 4 of the OpenAI Agents SDK section.

The goal is to understand how several focused agents can work together as one system before moving the implementation into modular Python files under `3_agents/deep-research-agent/`.

## Workflow

**User query → Planner → Search agents → Writer → Email**

The system uses four specialized agents:

1. **Search Agent** — searches the web for a specific query and summarizes results.
2. **Planner Agent** — turns the user's research question into a structured search plan.
3. **Writer Agent** — turns the collected research into a structured report.
4. **Email Agent** — uses a tool to deliver the completed report.

Python orchestrates the workflow by calling `Runner.run()` for each stage.


In [ ]:
from agents import Agent, WebSearchTool, trace, Runner, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio

load_dotenv(override=True)

MODEL_NAME = "gpt-5.4-mini"
HOW_MANY_SEARCHES = 5


# 1. Search Agent

**Responsibility:** search the web for a specific term and return a concise summary.

The hosted `WebSearchTool` gives this agent access to web search.


In [ ]:
SEARCH_INSTRUCTIONS = '''
You are a research assistant.

Given a search term, search the web for that term and
produce a concise summary of the results.
The summary should be 2-3 paragraphs and less than 300 words.

Capture the main points and be succinct.
Reply only with the summary.
'''

search_settings = ModelSettings(tool_choice="required")

search_agent = Agent(
    name="Search Agent",
    instructions=SEARCH_INSTRUCTIONS,
    tools=[WebSearchTool()],
    model=MODEL_NAME,
    model_settings=search_settings,
)


In [ ]:
task = "Most popular AI Agent frameworks in 2026"

result = await Runner.run(
    search_agent,
    task,
)

print(result.final_output)


# 2. Planner Agent

**Responsibility:** decide which searches should be performed to answer the user's question.

The planner does not perform the searches itself. It returns a structured `WebSearchPlan`.


In [ ]:
class WebSearchItem(BaseModel):
    reason: str = Field(
        description="Why this search is important for answering the query."
    )
    query: str = Field(
        description="The search term to use for the web search."
    )


class WebSearchPlan(BaseModel):
    searches: list[WebSearchItem] = Field(
        description=(
            "A list of web searches to perform "
            "to best answer the query."
        )
    )


In [ ]:
planner_instructions = f'''
You are a research assistant.

Given a user query, come up with a set of web searches
to perform to best answer the query.

Output {HOW_MANY_SEARCHES} search terms.
'''

planner_agent = Agent(
    name="Planner Agent",
    instructions=planner_instructions,
    model=MODEL_NAME,
    output_type=WebSearchPlan,
)

result = await Runner.run(
    planner_agent,
    task,
)

search_plan = result.final_output
print(search_plan)


# 3. Writer Agent

**Responsibility:** turn the original query and collected research into a structured report.


In [ ]:
class ReportData(BaseModel):
    short_summary: str = Field(
        description="A short 2-3 sentence summary of the findings."
    )
    markdown_report: str = Field(
        description="The final report in Markdown."
    )
    follow_up_questions: list[str] = Field(
        description="Suggested topics to research further."
    )


writer_instructions = '''
You are a senior researcher tasked with writing
a cohesive report for a research query.

You will be provided with the original query and research.

Generate a comprehensive report based on the research and query.
The final output should be in Markdown format.
'''

writer_agent = Agent(
    name="Writer Agent",
    instructions=writer_instructions,
    model=MODEL_NAME,
    output_type=ReportData,
)


# 4. Email Agent

**Responsibility:** deliver the completed report.

In the final modular application this tool is connected to the project's `messenger.py`. The notebook keeps the delivery function simple so the learning implementation can stand on its own.


In [ ]:
def send_email(subject: str, text_body: str, html_body: str):
    # Learning version: represent the external delivery action.
    print(f"Subject: {subject}")
    print(text_body)


def push(message: str):
    # Fallback representation of an external notification.
    print(f"Push: {message}")


@function_tool
def send_email_tool(
    subject: str,
    text_body: str,
    html_body: str,
) -> str:
    '''Send a completed research report.'''
    send_email(subject, text_body, html_body)
    return "Email sent successfully"


email_instructions = '''
You are provided with a detailed report.

Use your tool to send an email, converting the report
into a clean, well-presented HTML email with
an appropriate subject line.
'''

email_agent = Agent(
    name="Email Agent",
    instructions=email_instructions,
    tools=[send_email_tool],
    model=MODEL_NAME,
)


# 5. Orchestrating the Agents

The workflow is controlled by Python:

**Plan → Search → Write → Deliver**

The planned searches are independent, so the search stage can run them concurrently with `asyncio.gather()`.


In [ ]:
async def search(item: WebSearchItem):
    input_message = (
        f"Search term: {item.query}\n"
        f"Reason for searching: {item.reason}"
    )

    result = await Runner.run(
        search_agent,
        input_message,
    )

    return result.final_output


async def run_searches(query: str):
    print("Planning searches...")

    result = await Runner.run(
        planner_agent,
        f"Query: {query}",
    )

    searches = result.final_output.searches

    print(f"Will perform {len(searches)} searches")

    tasks = [search(item) for item in searches]
    results = await asyncio.gather(*tasks)

    print("Finished searching")

    return results


In [ ]:
async def write_report(
    query: str,
    search_results: list[str],
):
    print("Thinking about report...")

    input_message = (
        f"Original query: {query}\n"
        f"Summarized search results: {search_results}"
    )

    result = await Runner.run(
        writer_agent,
        input_message,
    )

    print("Finished writing report")

    return result.final_output


async def send_report_email(report: ReportData):
    print("Writing email...")

    result = await Runner.run(
        email_agent,
        report.markdown_report,
    )

    print("Email sent")

    return result.final_output


# 6. Running the Complete Workflow

The four agents now work together as one system.

A trace wraps the workflow so the full execution can be inspected.


In [ ]:
query = "Most popular AI Agent frameworks in 2026"

with trace("Research trace"):
    print("Starting research...")

    search_results = await run_searches(query)

    report = await write_report(
        query,
        search_results,
    )

    await send_report_email(report)

    print("Research complete.")

print(report.short_summary)


# How the agents collaborate

```text
User Query
    │
    ▼
Planner Agent
    │
    │  structured search plan
    ▼
Search Agents
    │
    │  research summaries
    ▼
Writer Agent
    │
    │  structured report
    ▼
Email Agent
    │
    ▼
External delivery
```

Each agent has one clear responsibility while Python coordinates the overall workflow.


# What I learned

The Deep Research Agent showed me how a larger agentic application can be broken into focused components.

The important lessons were:

- **Specialization:** each agent has one clear responsibility.
- **Structured Outputs:** agents can pass predictable data between stages.
- **Orchestration:** Python can control the workflow explicitly.
- **Parallel execution:** independent searches can run concurrently.
- **Tools:** an agent can perform an external action rather than only generate text.
- **Tracing:** the complete workflow can be inspected as one trace.

This is also where the modular architecture became important to me.

Instead of putting everything into one large Python file, the final implementation separates the agents, orchestration layer, messaging utilities, and application interface into their own files.


# From Learning Notebook to Project

The notebook documents how I learned the system.

The actual implementation is moved into:

```text
3_agents/
└── deep-research-agent/
    ├── app.py
    ├── research_manager.py
    ├── planner_agent.py
    ├── search_agent.py
    ├── writer_agent.py
    ├── email_agent.py
    ├── messenger.py
    └── styles.py
```

This keeps the **learning record** separate from the **modular application**.
